<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-classical-and-quantum-physics/blob/main/notebooks/lecture-12/lab-068t-quantum-toolbox.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Lab 068T — Quantum toolbox (recap of Lecture 11)

Every symbol used in Lecture 12 is a vector or a matrix. This notebook builds them with NumPy only, so you can check each example of the toolbox slides yourself.

**Route (about 20 minutes):** qubit states → Pauli matrices and expectation values → tensor products → Pauli strings → the two-spin Ising Hamiltonian and its ground state → gates and a circuit → the Bell state and the product test.

**Stopping point:** after Section 6 you can read every formula in the variational-circuit part of the lecture.

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1. A qubit is a vector of two amplitudes

$|0\rangle=(1,0)^T$, $|1\rangle=(0,1)^T$, and $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$. The Born rule gives $P(0)=|\alpha|^2$, $P(1)=|\beta|^2$. The bra $\langle\psi|$ is the conjugate row vector, so $\langle\phi|\psi\rangle$ is `np.vdot(phi, psi)`.

In [2]:
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
plus = (ket0 + ket1) / np.sqrt(2)

print("|+> =", plus)
print("P(0), P(1) =", np.abs(plus)**2)
print("<0|+> =", np.vdot(ket0, plus), "  |<0|+>|^2 =", abs(np.vdot(ket0, plus))**2)

|+> = [0.707+0.j 0.707+0.j]
P(0), P(1) = [0.5 0.5]
<0|+> = (0.7071067811865475+0j)   |<0|+>|^2 = 0.4999999999999999


## 2. Pauli matrices and expectation values

$\hat X$ flips the bit, $\hat Z$ gives $+1$ to $|0\rangle$ and $-1$ to $|1\rangle$. The expectation value $\langle\hat O\rangle=\langle\psi|\hat O|\psi\rangle$ is the average of the measured eigenvalues.

In [3]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def expval(op, psi):
    return np.vdot(psi, op @ psi).real

print("X|0> =", X @ ket0, "  Z|1> =", Z @ ket1)
print("X^2 = I:", np.allclose(X @ X, I), " Y^2 = I:", np.allclose(Y @ Y, I), " Z^2 = I:", np.allclose(Z @ Z, I))
for name, psi in [("|0>", ket0), ("|1>", ket1), ("|+>", plus)]:
    print(f"{name}:  <Z> = {expval(Z, psi):+.3f}   <X> = {expval(X, psi):+.3f}")

X|0> = [0.+0.j 1.+0.j]   Z|1> = [ 0.+0.j -1.+0.j]
X^2 = I: True  Y^2 = I: True  Z^2 = I: True
|0>:  <Z> = +1.000   <X> = +0.000
|1>:  <Z> = -1.000   <X> = +0.000
|+>:  <Z> = -0.000   <X> = +1.000


**Shots.** A device returns $\pm1$ outcomes, not $\langle\hat Z\rangle$. Averaging many outcomes estimates it, with an error that shrinks like $1/\sqrt{\text{shots}}$.

In [4]:
rng = np.random.default_rng(0)
theta = 1.0
psi = np.array([np.cos(theta/2), np.sin(theta/2)], dtype=complex)    # R_y(theta)|0>
p0 = abs(psi[0])**2
for shots in (10, 100, 1000, 10000):
    outcomes = np.where(rng.random(shots) < p0, +1, -1)
    print(f"{shots:6d} shots: estimate {outcomes.mean():+.3f}   exact {expval(Z, psi):+.3f}")

    10 shots: estimate +0.400   exact +0.540
   100 shots: estimate +0.400   exact +0.540
  1000 shots: estimate +0.534   exact +0.540
 10000 shots: estimate +0.541   exact +0.540


## 3. Two qubits: the tensor (Kronecker) product

`np.kron(a, b)` multiplies every entry of `a` by the whole vector `b`. The four entries of a two-qubit state belong to $|00\rangle,|01\rangle,|10\rangle,|11\rangle$, in this order; the first symbol is qubit 1.

In [5]:
ket01 = np.kron(ket0, ket1)
print("|01> =", ket01.real)
labels = ["00", "01", "10", "11"]
psi2 = np.kron(plus, ket1)          # |+> on qubit 1, |1> on qubit 2
for b, c in zip(labels, psi2):
    print(f"amplitude of |{b}> = {c.real:+.3f}   probability {abs(c)**2:.3f}")
print("N qubits need 2^N amplitudes; N = 50:", f"{2**50:.2e}")

|01> = [0. 1. 0. 0.]
amplitude of |00> = +0.000   probability 0.000
amplitude of |01> = +0.707   probability 0.500
amplitude of |10> = +0.000   probability 0.000
amplitude of |11> = +0.707   probability 0.500
N qubits need 2^N amplitudes; N = 50: 1.13e+15


## 4. Pauli strings

An operator on qubit $i$ alone is padded with identities: $\hat Z_1=\hat Z\otimes\hat I$, $\hat Z_2=\hat I\otimes\hat Z$. A product over several qubits, such as $\hat Z_1\hat Z_2=\hat Z\otimes\hat Z$, is a Pauli string. The helper below builds any string from a text such as `"XZZX"`.

In [6]:
P = {"I": I, "X": X, "Y": Y, "Z": Z}

def pauli_string(s):
    out = np.array([[1.0 + 0j]])
    for ch in s:
        out = np.kron(out, P[ch])
    return out

ZZ = pauli_string("ZZ")
print("Z1 Z2 = diag", np.diag(ZZ).real)
for b in labels:
    v = np.zeros(4, dtype=complex); v[int(b, 2)] = 1
    print(f"<{b}| Z1 Z2 |{b}> = {expval(ZZ, v):+.0f}")
print("X1 |01> = |11> :", np.allclose(pauli_string("XI") @ ket01, np.kron(ket1, ket1)))
print("shape of X Z Z X:", pauli_string("XZZX").shape)

Z1 Z2 = diag [ 1. -1. -1.  1.]
<00| Z1 Z2 |00> = +1
<01| Z1 Z2 |01> = -1
<10| Z1 Z2 |10> = -1
<11| Z1 Z2 |11> = +1
X1 |01> = |11> : True
shape of X Z Z X: (16, 16)


$\hat Z_1\hat Z_2$ is $+1$ for aligned and $-1$ for opposite spins: it measures the parity of the two bits.

## 5. A Hamiltonian is a sum of Pauli strings

Two-spin Ising model $\hat{\mathcal H}=-J\hat Z_1\hat Z_2-h(\hat X_1+\hat X_2)$. Its ground state is the eigenvector with the smallest eigenvalue.

In [7]:
def ising2(J, h):
    return -J * pauli_string("ZZ") - h * (pauli_string("XI") + pauli_string("IX"))

J, h = 1.0, 0.5
H = ising2(J, h)
print("H =\n", H.real)
E, V = np.linalg.eigh(H)
print("eigenvalues:", E)
g = V[:, 0]
print("ground state amplitudes:", g.real)
print("ground energy from <H>:", expval(H, g))
print("same from the Pauli terms: -J<ZZ> - h<X1> - h<X2> =",
      -J*expval(pauli_string("ZZ"), g) - h*expval(pauli_string("XI"), g) - h*expval(pauli_string("IX"), g))

H =
 [[-1.  -0.5 -0.5 -0. ]
 [-0.5  1.  -0.  -0.5]
 [-0.5 -0.   1.  -0.5]
 [-0.  -0.5 -0.5 -1. ]]
eigenvalues: [-1.414 -1.     1.     1.414]
ground state amplitudes: [-0.653 -0.271 -0.271 -0.653]
ground energy from <H>: -1.414213562373094
same from the Pauli terms: -J<ZZ> - h<X1> - h<X2> = -1.414213562373094


In [8]:
# limiting cases from the slide
print("h = 0: lowest eigenvalues", np.linalg.eigh(ising2(1, 0))[0][:2], "  (|00>, |11>, E = -J)")
print("J = 0: lowest eigenvalue ", np.linalg.eigh(ising2(0, 1))[0][0], "  (|++>, E = -2h)")

h = 0: lowest eigenvalues [-1. -1.]   (|00>, |11>, E = -J)
J = 0: lowest eigenvalue  -2.0   (|++>, E = -2h)


## 6. Gates are unitary matrices, circuits are their products

$\hat R_y(\theta)$ rotates the qubit, $\hat H$ makes $|+\rangle$, CNOT flips qubit 2 if qubit 1 is $|1\rangle$, CZ puts a minus sign on $|11\rangle$. In a circuit, time runs left to right, so the matrices multiply right to left.

In [9]:
def Ry(t):
    return np.array([[np.cos(t/2), -np.sin(t/2)], [np.sin(t/2), np.cos(t/2)]], dtype=complex)
Hd = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
CZ = np.diag([1, 1, 1, -1]).astype(complex)

print("Ry(pi/2)|0> =", (Ry(np.pi/2) @ ket0).real, "= |+>")
print("Ry(pi)|0>   =", (Ry(np.pi) @ ket0).real, "= |1>")
print("unitary: U^dagger U = I ->", np.allclose(CNOT.conj().T @ CNOT, np.eye(4)))

# a small trial circuit on two qubits: Ry on each qubit, then CZ
def trial_state(t1, t2):
    return CZ @ np.kron(Ry(t1), Ry(t2)) @ np.kron(ket0, ket0)

psi = trial_state(0.4, 1.1)
print("trial state:", psi.real, "  energy <H> =", round(expval(H, psi), 4), " >= ground", round(E[0], 4))

Ry(pi/2)|0> = [0.707 0.707] = |+>
Ry(pi)|0>   = [0. 1.] = |1>
unitary: U^dagger U = I -> True
trial state: [ 0.836  0.512  0.169 -0.104]   energy <H> = -0.9165  >= ground -1.4142


Every trial state has an energy above the ground energy. Lecture 12 trains the angles of such circuits to push the energy down: that is the variational quantum eigensolver.

## 7. Entanglement: the Bell state and the product test

In [10]:
bell = CNOT @ np.kron(Hd, I) @ np.kron(ket0, ket0)
print("Bell state:", bell.real)
print("<Z1> =", expval(pauli_string("ZI"), bell), "  <Z1 Z2> =", expval(pauli_string("ZZ"), bell))

def is_product(psi2):
    C = psi2.reshape(2, 2)          # rows: qubit 1, columns: qubit 2
    return abs(np.linalg.det(C)) < 1e-12, np.linalg.det(C)

print("Bell  product?", is_product(bell))
print("|+0>  product?", is_product(np.kron(plus, ket0)))

Bell state: [0.707 0.    0.    0.707]
<Z1> = -2.2371143170757382e-17   <Z1 Z2> = 0.9999999999999998
Bell  product? (np.False_, np.complex128(0.4999999999999999+0j))
|+0>  product? (np.True_, np.complex128(0j))


Each qubit of the Bell state gives 0 or 1 at random ($\langle\hat Z_1\rangle=0$), but the results always agree ($\langle\hat Z_1\hat Z_2\rangle=+1$), and $\det C\neq0$ shows that no product of one-qubit states can produce it.

## Exercises
1. Compute $\langle\hat Z_1\hat Z_2\rangle$ and $\langle\hat X_1\rangle$ for the ground state of Section 5 at $h=0.1$ and $h=3$. Which term wins in each limit?
2. Build the three-spin Ising Hamiltonian $-\sum_{i=1}^{2}\hat Z_i\hat Z_{i+1}-h\sum_{i=1}^{3}\hat X_i$ with `pauli_string` and find its ground energy at $h=1$.
3. Scan the two angles of `trial_state` on a grid and find the lowest energy. How close does it get to `E[0]`?